----
# ML-DSA
----

In [1]:
import os
import time
import csv
import tracemalloc
import numpy as np
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.serialization import Encoding, NoEncryption, PrivateFormat, PublicFormat

# Use the locally built liboqs install if available
oqs_install_path = r"C:\Users\Admin\_oqs_0.14.0"
if os.path.isdir(oqs_install_path):
    os.environ["OQS_INSTALL_PATH"] = oqs_install_path
    os.environ["PATH"] = os.environ.get("PATH", "") + ";" + os.path.join(oqs_install_path, "lib")
    print("Using OQS_INSTALL_PATH:", os.environ["OQS_INSTALL_PATH"])

import oqs

# Configuration
message = b"Benchmarking Post-Quantum Cryptographic Algorithms"

# File paths for ML-DSA
ml_dsa_benchmark_output = "ml_dsa_benchmark_results.txt"
ml_dsa_csv_output = "ml_dsa_timing.csv"

# File paths for RSA-3072
rsa_3072_benchmark_output = "rsa_3072_benchmark_results.txt"
rsa_3072_csv_output = "rsa_3072_timing.csv"

def write_header(file_path, header_text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(header_text + "\n")
        f.write("=" * len(header_text) + "\n")

write_header(ml_dsa_benchmark_output, "ML-DSA Benchmark Results")
write_header(rsa_3072_benchmark_output, "RSA-3072 Benchmark Results")

csv_enabled = True
try:
    with open(ml_dsa_csv_output, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
    with open(rsa_3072_csv_output, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
except PermissionError:
    print(f"Warning: Cannot write to one or more CSV files (file may be open in another program). CSV logging disabled.")
    csv_enabled = False

def append_to_output(file_path, *lines):
    with open(file_path, "a", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")

def append_to_csv(csv_path, metric, run, time_ms, peak_kb):
    if csv_enabled:
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow([metric, run, time_ms, peak_kb])

# ==========================================
# PART 1: ML-DSA-44
# ==========================================
alg = "ML-DSA-44"
print(f"Benchmarking {alg}...")

# Keypair generation benchmark
print(f"Benchmarking {alg} keypair generation...")
print("Warming up...")
with oqs.Signature(alg) as sig:
    for _ in range(100):
        _ = sig.generate_keypair()

runs = 10000
ml_keypair_times = []
ml_keypair_peaks = []
with oqs.Signature(alg) as sig:
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        _ = sig.generate_keypair()
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        ml_keypair_times.append(elapsed_ms)
        ml_keypair_peaks.append(peak_kb)
        append_to_csv(ml_dsa_csv_output, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(ml_keypair_times):.3f} ms")
print(f"Min keypair time: {np.min(ml_keypair_times):.3f} ms")
print(f"Max keypair time: {np.max(ml_keypair_times):.3f} ms")
print(f"Mean peak memory: {np.mean(ml_keypair_peaks):.3f} KB")
print(f"Max peak memory: {np.max(ml_keypair_peaks):.3f} KB")

append_to_output(
    ml_dsa_benchmark_output,
    f"\nBenchmarking {alg} keypair generation...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(ml_keypair_times):.3f} ms",
    f"Min keypair time: {np.min(ml_keypair_times):.3f} ms",
    f"Max keypair time: {np.max(ml_keypair_times):.3f} ms",
    f"Mean peak memory: {np.mean(ml_keypair_peaks):.3f} KB",
    f"Max peak memory: {np.max(ml_keypair_peaks):.3f} KB")

# Signing benchmark
print(f"Benchmarking {alg} signing...")
print("Warming up...")
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    context = b""
    for _ in range(100):
        _ = sig.sign_with_ctx_str(message, context)

ml_signing_times = []
ml_signing_peaks = []
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    context = b""
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        signature = sig.sign_with_ctx_str(message, context)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        ml_signing_times.append(elapsed_ms)
        ml_signing_peaks.append(peak_kb)
        append_to_csv(ml_dsa_csv_output, "signing", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean signing time: {np.mean(ml_signing_times):.3f} ms")
print(f"Min signing time: {np.min(ml_signing_times):.3f} ms")
print(f"Max signing time: {np.max(ml_signing_times):.3f} ms")
print(f"Mean peak memory: {np.mean(ml_signing_peaks):.3f} KB")
print(f"Max peak memory: {np.max(ml_signing_peaks):.3f} KB")

append_to_output(
    ml_dsa_benchmark_output,
    f"\nBenchmarking {alg} signing...",
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean signing time: {np.mean(ml_signing_times):.3f} ms",
    f"Min signing time: {np.min(ml_signing_times):.3f} ms",
    f"Max signing time: {np.max(ml_signing_times):.3f} ms",
    f"Mean peak memory: {np.mean(ml_signing_peaks):.3f} KB",
    f"Max peak memory: {np.max(ml_signing_peaks):.3f} KB")

# Size metrics
print(f"Measuring {alg} size metrics...")
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    sk = sig.export_secret_key()
    context = b""
    signature = sig.sign_with_ctx_str(message, context)

print(f"Algorithm: {alg}")
print(f"Public key size: {len(pk)} bytes")
print(f"Secret key size: {len(sk)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    ml_dsa_benchmark_output,
    f"\nMeasuring {alg} size metrics...",
    f"Algorithm: {alg}",
    f"Public key size: {len(pk)} bytes",
    f"Secret key size: {len(sk)} bytes",
    f"Signature size: {len(signature)} bytes")

# ==========================================
# PART 2: RSA-3072
# ==========================================
print("\nBenchmarking RSA-3072...")

# Keypair generation benchmark
print("Benchmarking RSA-3072 keypair generation...")
print("Warming up...")
for _ in range(100):
    _ = rsa.generate_private_key(public_exponent=65537, key_size=3072)

rsa_keygen_times = []
rsa_keygen_peaks = []
for run_index in range(runs):
    start = time.perf_counter()
    tracemalloc.start()
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    rsa_keygen_times.append(elapsed_ms)
    rsa_keygen_peaks.append(peak_kb)
    append_to_csv(rsa_3072_csv_output, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms")
print(f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms")
print(f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms")
print(f"Mean peak memory: {np.mean(rsa_keygen_peaks):.3f} KB")
print(f"Max peak memory: {np.max(rsa_keygen_peaks):.3f} KB")

append_to_output(
    rsa_3072_benchmark_output,
    "\nBenchmarking RSA-3072 keypair generation...",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms",
    f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms",
    f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms",
    f"Mean peak memory: {np.mean(rsa_keygen_peaks):.3f} KB",
    f"Max peak memory: {np.max(rsa_keygen_peaks):.3f} KB")

# Signing benchmark
print("Benchmarking RSA-3072 signing...")
print("Warming up...")
test_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
for _ in range(100):
    _ = test_key.sign(
        message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )

rsa_signing_times = []
rsa_signing_peaks = []
for run_index in range(runs):
    start = time.perf_counter()
    tracemalloc.start()
    signature = private_key.sign(
        message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    rsa_signing_times.append(elapsed_ms)
    rsa_signing_peaks.append(peak_kb)
    append_to_csv(rsa_3072_csv_output, "signing", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {runs}")
print(f"Mean signing time: {np.mean(rsa_signing_times):.3f} ms")
print(f"Min signing time: {np.min(rsa_signing_times):.3f} ms")
print(f"Max signing time: {np.max(rsa_signing_times):.3f} ms")
print(f"Mean peak memory: {np.mean(rsa_signing_peaks):.3f} KB")
print(f"Max peak memory: {np.max(rsa_signing_peaks):.3f} KB")

append_to_output(
    rsa_3072_benchmark_output,
    "\nBenchmarking RSA-3072 signing...",
    f"Runs: {runs}",
    f"Mean signing time: {np.mean(rsa_signing_times):.3f} ms",
    f"Min signing time: {np.min(rsa_signing_times):.3f} ms",
    f"Max signing time: {np.max(rsa_signing_times):.3f} ms",
    f"Mean peak memory: {np.mean(rsa_signing_peaks):.3f} KB",
    f"Max peak memory: {np.max(rsa_signing_peaks):.3f} KB")

# Size metrics
print("Measuring RSA-3072 size metrics...")
rsa_public_key = private_key.public_key()
rsa_public_bytes = rsa_public_key.public_bytes(
    encoding=Encoding.PEM,
    format=PublicFormat.SubjectPublicKeyInfo,
)
rsa_private_bytes = private_key.private_bytes(
    encoding=Encoding.PEM,
    format=PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=NoEncryption(),
)

print(f"Public key size: {len(rsa_public_bytes)} bytes")
print(f"Private key size: {len(rsa_private_bytes)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    rsa_3072_benchmark_output,
    "\nMeasuring RSA-3072 size metrics...",
    f"Public key size: {len(rsa_public_bytes)} bytes",
    f"Private key size: {len(rsa_private_bytes)} bytes",
    f"Signature size: {len(signature)} bytes")

print("\nBenchmarking complete. All files have been generated.")

Using OQS_INSTALL_PATH: C:\Users\Admin\_oqs_0.14.0
Benchmarking ML-DSA-44...
Benchmarking ML-DSA-44 keypair generation...
Warming up...
Algorithm: ML-DSA-44
Runs: 10000
Mean keypair time: 0.928 ms
Min keypair time: 0.577 ms
Max keypair time: 18.102 ms
Mean peak memory: 5.329 KB
Max peak memory: 8.206 KB
Benchmarking ML-DSA-44 signing...
Warming up...
Algorithm: ML-DSA-44
Runs: 10000
Mean signing time: 3.566 ms
Min signing time: 0.923 ms
Max signing time: 29.380 ms
Mean peak memory: 5.464 KB
Max peak memory: 6.120 KB
Measuring ML-DSA-44 size metrics...
Algorithm: ML-DSA-44
Public key size: 1312 bytes
Secret key size: 2560 bytes
Signature size: 2420 bytes

Benchmarking RSA-3072...
Benchmarking RSA-3072 keypair generation...
Warming up...
Runs: 10000
Mean keypair time: 657.897 ms
Min keypair time: 25.925 ms
Max keypair time: 4872.525 ms
Mean peak memory: 0.074 KB
Max peak memory: 18.639 KB
Benchmarking RSA-3072 signing...
Warming up...
Runs: 10000
Mean signing time: 5.264 ms
Min signing t

In [2]:
import os
import oqs

# Test import
print("oqs imported successfully")
print("Available signature algorithms:", oqs.get_enabled_sig_mechanisms())

oqs imported successfully
Available signature algorithms: ('Dilithium2', 'Dilithium3', 'Dilithium5', 'ML-DSA-44', 'ML-DSA-65', 'ML-DSA-87', 'Falcon-512', 'Falcon-1024', 'Falcon-padded-512', 'Falcon-padded-1024', 'SPHINCS+-SHA2-128f-simple', 'SPHINCS+-SHA2-128s-simple', 'SPHINCS+-SHA2-192f-simple', 'SPHINCS+-SHA2-192s-simple', 'SPHINCS+-SHA2-256f-simple', 'SPHINCS+-SHA2-256s-simple', 'SPHINCS+-SHAKE-128f-simple', 'SPHINCS+-SHAKE-128s-simple', 'SPHINCS+-SHAKE-192f-simple', 'SPHINCS+-SHAKE-192s-simple', 'SPHINCS+-SHAKE-256f-simple', 'SPHINCS+-SHAKE-256s-simple', 'MAYO-1', 'MAYO-2', 'MAYO-3', 'MAYO-5', 'cross-rsdp-128-balanced', 'cross-rsdp-128-fast', 'cross-rsdp-128-small', 'cross-rsdp-192-balanced', 'cross-rsdp-192-fast', 'cross-rsdp-192-small', 'cross-rsdp-256-balanced', 'cross-rsdp-256-fast', 'cross-rsdp-256-small', 'cross-rsdpg-128-balanced', 'cross-rsdpg-128-fast', 'cross-rsdpg-128-small', 'cross-rsdpg-192-balanced', 'cross-rsdpg-192-fast', 'cross-rsdpg-192-small', 'cross-rsdpg-256-b